In [7]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()
# walks upward until it finds a directory containing ppm/
PROJECT_ROOT = next(
    (p for p in (current_dir, *current_dir.parents) if (p / "ppm").is_dir()),
    current_dir,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from ppm.wandb_utils import load_multiple_experiments

plots_dir = PROJECT_ROOT / "plots/pre-training"
plots_dir.mkdir(parents=True, exist_ok=True)

In [8]:
PROJECTS=[
  "BPI12_001",
  "BPI12_002",
  "BPI12_003",
  "Distill_BPI12_001",
  "BPI15_001",
  "BPI15_002",
  "BPI15_003",
  "Distill_BPI15_001",
  "BPI17_001",
  "Distill_BPI17_001",
  "BPI17_002",
  "BPI17_003",
  "BPI19_001",
  "BPI19_002",
  "BPI19_003",
  "Distill_BPI19_001",  
  "BPI20_001",
  "BPI20_002",
  "BPI20_003",
  "Distill_BPI20_PTC_001",
  "Distill_BPI20_RFP_001",
  "Distill_BPI20_TPD_001",
  "Distill_BPI20_PTC_002",
  "Distill_BPI20_RFP_002",
  "Distill_BPI20_TPD_002",
  "LSTM_001",
  "baseline-nep"
]

runs_raw, _ = load_multiple_experiments(PROJECTS, force_update=False)

runs_raw["best_test_final_next_activity_f1"] = (
    runs_raw["best_test_final_next_activity_f1"]
    .combine_first(runs_raw["best_test_final_next_activity_f1_macro"])
)
#runs_raw = runs_raw.drop(columns=["best_test_final_next_activity_f1_macro"])



Database already exists: /app/visualization/paper/metrics/BPI12_001.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI12_002.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI12_003.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/Distill_BPI12_001.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI15_001.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI15_002.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI15_003.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/Distill_BPI15_001.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metric

In [9]:
import pandas as pd

pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.width", None)         # don't wrap/truncate by width
pd.set_option("display.max_colwidth", None)  # optional: full cell content

runs_raw.columns

Index(['id', 'name', 'r', 'lr', 'log', 'device', 'epochs', 'compile',
       'n_heads', 'backbone',
       ...
       'train_next_activity_CE_loss', 'train_next_activity_KL_loss',
       'train_next_activity_total_loss', 'val_next_activity_CE_loss',
       'val_next_activity_KL_loss', 'val_next_activity_total_loss', 'seed',
       'prediction_table', 'transition_table',
       'best_test_final_next_activity_f1_macro'],
      dtype='str', length=118)

In [10]:
# prepocessing
runs = runs_raw.copy()

EXCLUDE = ["LSTM-256-1", "LSTM-128-1"]

fine_tuning_col = "fine_tuning" if "fine_tuning" in runs.columns else "fine-tuning"
freeze_layers_col = "freeze_layers" if "freeze_layers" in runs.columns else "freeze-layers"
mask_no_finetuning = runs[fine_tuning_col].isna() | (runs[fine_tuning_col].astype(str) == "None")
runs.loc[mask_no_finetuning, freeze_layers_col] = "all"

for col in ["hidden_size", "n_layers", "n_heads"]:
    if col in runs.columns:
        runs[col] = pd.to_numeric(runs[col], errors="coerce")

def _fmt_int(v):
    return str(int(v)) if pd.notna(v) else "nan"

def _arch_label(df: pd.DataFrame) -> pd.Series:
    hs = df["hidden_size"].map(_fmt_int).astype("string")
    nl = df["n_layers"].map(_fmt_int).astype("string")
    nh = df["n_heads"].map(_fmt_int).astype("string")
    return hs.str.cat([nl, nh], sep="-")

def _lstm_label(df: pd.DataFrame) -> pd.Series:
    hs = df["hidden_size"].map(_fmt_int).astype("string")
    nl = df["n_layers"].map(_fmt_int).astype("string")
    return hs.str.cat(nl, sep="-")

# Identify distill and scratch (nano) runs separately
lstm_mask = runs["project"].astype(str).str.startswith("LSTM_00", na=False)
distill_mask = runs["project"].astype(str).str.startswith("Distill", na=False)
scratch_mask = (
    runs["backbone"].astype(str).str.contains("student", na=False)
    & ~distill_mask
)

runs.loc[lstm_mask, "backbone"] = "LSTM-" + _lstm_label(runs.loc[lstm_mask])
runs.loc[scratch_mask, "backbone"] = "nano-" + _arch_label(runs.loc[scratch_mask])
runs.loc[distill_mask, "backbone"] = "distill-" + _arch_label(runs.loc[distill_mask])

backbone_rename_map = {
    "BPI20PrepaidTravelCosts": "BPI20_PTC",
    "BPI20TravelPermitData": "BPI20_TPD",
    "BPI20RequestForPayment": "BPI20_RfP",
}
runs["log"] = runs["log"].replace(backbone_rename_map)
runs = runs[~runs["backbone"].isin(EXCLUDE)].copy()


## Show data

In [11]:
# Average wall-clock time per model, balanced across datasets
wall_clock = runs.dropna(subset=["duration_sec", "backbone", "log"]).copy()

per_dataset = (
    wall_clock.groupby(["backbone", "log"], as_index=False)
    .agg(
        duration_sec=("duration_sec", "mean"),
        epochs=("epochs", "mean"),
        n_runs=("duration_sec", "size"),
    )
)

wall_clock_summary = (
    per_dataset.groupby("backbone")
    .agg(
        datasets=("log", "nunique"),
        total_runs=("n_runs", "sum"),
        avg_minutes=("duration_sec", lambda s: s.mean() / 60),
        min_minutes=("duration_sec", lambda s: s.min() / 60),
        max_minutes=("duration_sec", lambda s: s.max() / 60),
        std_minutes=("duration_sec", lambda s: s.std() / 60),
        avg_epochs=("epochs", "mean"),
        min_epochs=("epochs", "min"),
        max_epochs=("epochs", "max"),
    )
    .sort_values("avg_minutes")
    .reset_index()
)

wall_clock_summary.round(2)


,backbone,datasets,total_runs,avg_minutes,min_minutes,max_minutes,std_minutes
0,baseline_transition_frequency,7,15,0.03,0.02,0.04,0.01
1,nano-64-2-1,7,35,1.47,0.13,7.23,2.56
2,nano-64-4-1,7,34,1.96,0.20,10.07,3.60
3,nano-128-4-2,7,38,2.04,0.21,9.75,3.44
4,LSTM-64-1,7,29,2.24,0.43,8.31,2.83
5,nano-256-4-4,7,35,2.44,0.24,11.06,3.86
6,LSTM-128-2,7,35,2.47,0.32,10.06,3.44
7,LSTM-256-2,7,35,2.58,0.24,11.19,3.91
8,nano-768-4-12,7,35,2.88,0.48,10.62,3.59
9,gpt2-mini,7,35,3.01,0.45,11.36,3.89


In [12]:
# Average wall-clock time per model family, balanced across datasets
def model_family(backbone: str) -> str:
    backbone = str(backbone)
    if backbone.startswith("qwen"):
        return "qwen"
    if backbone.startswith("nano"):
        return "nano"
    if backbone.startswith("distill"):
        return "distill"
    if backbone.startswith("gpt2"):
        return "gpt2"
    if backbone.startswith("LSTM"):
        return "LSTM"
    return "baseline"

per_dataset_family = per_dataset.assign(family=per_dataset["backbone"].map(model_family))

wall_clock_family_summary = (
    per_dataset_family.groupby(["family", "log"], as_index=False)
    .agg(
        duration_sec=("duration_sec", "mean"),
        epochs=("epochs", "mean"),
        models=("backbone", "nunique"),
    )
    .groupby("family")
    .agg(
        datasets=("log", "nunique"),
        avg_minutes=("duration_sec", lambda s: s.mean() / 60),
        min_minutes=("duration_sec", lambda s: s.min() / 60),
        max_minutes=("duration_sec", lambda s: s.max() / 60),
        std_minutes=("duration_sec", lambda s: s.std() / 60),
        avg_epochs=("epochs", "mean"),
        min_epochs=("epochs", "min"),
        max_epochs=("epochs", "max"),
    )
    .sort_values("avg_minutes")
    .reset_index()
)

wall_clock_family_summary.round(2)


,family,datasets,avg_minutes,min_minutes,max_minutes,std_minutes
0,baseline,7,0.03,0.02,0.04,0.01
1,LSTM,7,2.65,0.32,11.05,3.82
2,nano,7,3.96,0.58,15.31,5.16
3,distill,7,12.38,0.43,38.21,15.59
4,gpt2,7,13.67,1.91,56.83,19.67
5,qwen,7,66.57,4.17,281.03,100.37
